# Mini-Beispiel: Linear-Chain Conditional Random Field

Wir labeln die Sequenz `Anna lebt in Berlin` mit `PER`, `LOC` oder `O`. Das Notebook zeigt bewusst die **Inferenz eines bereits parametrisierten CRF**. Die Matrizen werden nicht trainiert, sondern so gewählt, dass jede Rechengröße sichtbar bleibt.

In [ ]:
import itertools
import numpy as np
import pandas as pd

labels=np.array(["PER","LOC","O"])
label_id={label:i for i,label in enumerate(labels)}
feature_namen=["gross", "nach_in", "verb", "ist_in", "bias"]

# Zeilen: PER, LOC, O; Spalten: die fünf Features
W=np.array([
    [ 1.4, -1.0, -1.0, -0.5,  0.0],
    [ 1.0,  1.8, -1.0, -0.5, -0.2],
    [-0.8, -0.4,  1.2,  1.4,  0.5],
])

# Zeile = vorheriges Label, Spalte = aktuelles Label
A=np.array([
    [ 0.4, -0.6, 0.7],
    [-0.5,  0.3, 0.6],
    [ 0.2,  1.1, 0.8],
])

def features(tokens):
    result=[]
    for t,token in enumerate(tokens):
        result.append([
            float(token[0].isupper()),
            float(t>0 and tokens[t-1].lower()=="in"),
            float(token.lower() in {"lebt","wohnt","arbeitet"}),
            float(token.lower()=="in"),
            1.0,
        ])
    return np.array(result)

def emissions(tokens):
    F=features(tokens)
    return F@W.T

def sequenz_score(E,sequenz,A_matrix=A):
    ids=[label_id[x] for x in sequenz]
    lokal=sum(E[t,k] for t,k in enumerate(ids))
    transition=sum(A_matrix[ids[t-1],ids[t]] for t in range(1,len(ids)))
    return lokal+transition

def logsumexp(x,axis=None):
    maximum=np.max(x,axis=axis,keepdims=True)
    wert=maximum+np.log(np.sum(np.exp(x-maximum),axis=axis,keepdims=True))
    return np.squeeze(wert,axis=axis) if axis is not None else float(wert.squeeze())

## 1. Vom Token zum lokalen Score-Vektor

Für jedes Token entsteht ein Feature-Vektor `f_t` mit fünf Einträgen. Dieselbe Matrix `W` mit der Form `(3, 5)` erzeugt an jeder Position den Vektor `e_t = W @ f_t`. Im Code stehen die Positionen in Zeilen, deshalb rechnen wir äquivalent `F @ W.T`. Das Ergebnis `E` hat die Form `(T, K)`.

In [ ]:
tokens=["Anna","lebt","in","Berlin"]
F=features(tokens); E=emissions(tokens)
display(pd.DataFrame(F,index=tokens,columns=feature_namen))
display(pd.DataFrame(E,index=tokens,columns=labels).round(2))
print("F:",F.shape,"W:",W.shape,"E:",E.shape)

## 2. Eine konkrete Kandidatensequenz bewerten

Aus jedem lokalen Score-Vektor wird genau der Eintrag des angenommenen Labels ausgewählt. Dazu kommen drei Übergangsscores aus `A`. Die Summe ist der skalare Sequenzscore `S(x, y)`.

In [ ]:
kandidat=["PER","O","O","LOC"]
ids=[label_id[x] for x in kandidat]
beitraege=[]
for t,(token,label,k) in enumerate(zip(tokens,kandidat,ids)):
    beitraege.append({"Position":t,"Token":token,"Label":label,"lokaler Score":E[t,k],"Transition davor":0 if t==0 else A[ids[t-1],k]})
tabelle=pd.DataFrame(beitraege)
display(tabelle.round(2))
print("Sequenzscore:",round(sequenz_score(E,kandidat),2))

## 3. Aus Scores werden Wahrscheinlichkeiten

Bei nur `K**T = 3**4 = 81` Sequenzen können wir zur Kontrolle alle aufzählen. In echten Aufgaben nutzt man dynamische Programmierung. Die stabile `logsumexp`-Berechnung verhindert numerisches Überlaufen.

In [ ]:
alle=list(itertools.product(labels,repeat=len(tokens)))
scores=np.array([sequenz_score(E,s) for s in alle])
logZ=logsumexp(scores)
wahrscheinlichkeiten=np.exp(scores-logZ)
ranking=pd.DataFrame({"Sequenz":[" ".join(s) for s in alle],"Score":scores,"P(y|x)":wahrscheinlichkeiten}).sort_values("P(y|x)",ascending=False)
display(ranking.head(8).round(4)); print("Summe der Wahrscheinlichkeiten:",wahrscheinlichkeiten.sum())

## 4. Forward berechnet `log Z(x)` ohne Aufzählung

Forward fasst alle Pfade zusammen, die an einer Position in demselben Label enden. Die Laufzeit ist `O(T * K**2)` statt `O(K**T)`.

In [ ]:
def forward_logZ(E,A_matrix=A):
    alpha=E[0].copy()
    for t in range(1,len(E)):
        # vorheriges Label in den Zeilen, aktuelles Label in den Spalten
        alpha=E[t]+logsumexp(alpha[:,None]+A_matrix,axis=0)
    return logsumexp(alpha)

print("log Z durch Aufzählung:",round(logZ,8))
print("log Z durch Forward:   ",round(forward_logZ(E),8))

## 5. Viterbi findet die beste Sequenz

Viterbi ersetzt `logsumexp` durch `max` und merkt sich Rückzeiger. So wird der beste vollständige Pfad rekonstruiert.

In [ ]:
def viterbi(E,A_matrix=A):
    delta=E[0].copy()
    rueckzeiger=[]
    for t in range(1,len(E)):
        kandidaten=delta[:,None]+A_matrix
        rueckzeiger.append(np.argmax(kandidaten,axis=0))
        delta=E[t]+np.max(kandidaten,axis=0)
    pfad=[int(np.argmax(delta))]
    for bp in reversed(rueckzeiger):
        pfad.append(int(bp[pfad[-1]]))
    pfad=pfad[::-1]
    return labels[pfad].tolist(),float(np.max(delta))

beste,bester_score=viterbi(E)
print("Viterbi:",list(zip(tokens,beste)),"Score:",round(bester_score,2))
print("Brute Force:",ranking.iloc[0]["Sequenz"],"Score:",round(ranking.iloc[0]["Score"],2))

## 6. Warum variable Längen funktionieren

`W` und `A` bleiben gleich; nur die Anzahl ihrer Anwendungen ändert sich. Dasselbe CRF kann daher eine neue Sequenz anderer Länge auswerten.

In [ ]:
for neue_tokens in [["Tom","wohnt","in","Rom"],["Eva","arbeitet","in","New","York"]]:
    neue_labels,score=viterbi(emissions(neue_tokens))
    print(list(zip(neue_tokens,neue_labels)),"Score:",round(score,2))

## Was hier noch nicht passiert

Ein echtes Training lernt `W` und `A`, indem es die negative Log-Likelihood `-S(x, y*) + log Z(x)` minimiert. Dieses kurze Beispiel konzentriert sich auf die Größen und Algorithmen aus der Unterlage: lokale Scores, Transitionen, Forward und Viterbi.